# Instruction Tuning

## Historical Context

Instruction tuning transformed LLMs from next-token predictors to helpful assistants:

- **2021**: FLAN (Fine-tuned LAnguage Net) by Google - first large-scale instruction tuning
- **2021**: T0 (BigScience) - multitask prompted training
- **2022**: InstructGPT (OpenAI) - RLHF on instruction following
- **2022**: FLAN-T5 - instruction-tuned T5 models
- **2023 March**: Alpaca (Stanford) - instruction tuning LLaMA 7B with 52K examples
- **2023 April**: Vicuna - trained on ShareGPT conversations
- **2023 April**: Dolly 2.0 (Databricks) - 15K human-generated instructions
- **2023**: Orca, WizardLM - synthetic instruction generation
- **2024-2025**: Mixture of instruction datasets, multi-turn dialogues, tool use

## Why Instruction Tuning Matters

**Base model** (pre-trained only):
- Completes text naturally
- User: "Write a poem" → Model: "about dogs, cats, trees..."

**Instruction-tuned model**:
- Follows instructions
- User: "Write a poem" → Model: *[writes a poem]*

**Key improvements**:
- Zero-shot task performance
- Better instruction following
- More helpful, harmless, honest
- Understands user intent

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import json
from typing import List, Dict, Optional
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass

## 1. Instruction Formats

Different formats for different use cases

In [ ]:
@dataclass
class InstructionExample:
    """Base class for instruction examples."""
    instruction: str
    input: str
    output: str


class AlpacaFormat:
    """Alpaca instruction format (Stanford, 2023).
    
    Format:
        Below is an instruction that describes a task, paired with an input...
        
        ### Instruction:
        {instruction}
        
        ### Input:
        {input}
        
        ### Response:
        {output}
    
    Used by: Alpaca, Alpaca-LoRA, many open-source models
    """
    
    PROMPT_TEMPLATE = (
        "Below is an instruction that describes a task, paired with an input that provides further context. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{instruction}\n\n"
        "### Input:\n{input}\n\n"
        "### Response:\n"
    )
    
    PROMPT_TEMPLATE_NO_INPUT = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{instruction}\n\n"
        "### Response:\n"
    )
    
    @classmethod
    def format(cls, example: InstructionExample) -> str:
        """Format example as Alpaca-style prompt."""
        if example.input:
            prompt = cls.PROMPT_TEMPLATE.format(
                instruction=example.instruction,
                input=example.input
            )
        else:
            prompt = cls.PROMPT_TEMPLATE_NO_INPUT.format(
                instruction=example.instruction
            )
        return prompt + example.output


class DollyFormat:
    """Dolly instruction format (Databricks, 2023).
    
    Similar to Alpaca but with category labels.
    Categories: open_qa, closed_qa, summarization, information_extraction, etc.
    """
    
    PROMPT_TEMPLATE = (
        "Instruction:\n{instruction}\n\n"
        "Context:\n{context}\n\n"
        "Response:\n"
    )
    
    @classmethod
    def format(cls, instruction: str, context: str = "", response: str = "") -> str:
        prompt = f"Instruction:\n{instruction}\n\n"
        if context:
            prompt += f"Context:\n{context}\n\n"
        prompt += "Response:\n"
        return prompt + response


class ShareGPTFormat:
    """ShareGPT conversation format (multi-turn dialogues).
    
    Format: List of messages with roles (user, assistant, system)
    Used by: Vicuna, many chat models
    
    Example:
        [
            {"role": "user", "content": "Hello!"},
            {"role": "assistant", "content": "Hi! How can I help?"},
            {"role": "user", "content": "Tell me a joke"},
            {"role": "assistant", "content": "Why did the..."},
        ]
    """
    
    @classmethod
    def format(cls, messages: List[Dict[str, str]], 
               system_prompt: Optional[str] = None) -> str:
        """Format multi-turn conversation."""
        formatted = ""
        
        if system_prompt:
            formatted += f"System: {system_prompt}\n\n"
        
        for msg in messages:
            role = msg["role"].capitalize()
            content = msg["content"]
            formatted += f"{role}: {content}\n\n"
        
        return formatted.strip()


class ChatMLFormat:
    """ChatML format (OpenAI-style with special tokens).
    
    Uses special tokens to delineate messages:
    <|im_start|>system\n{content}<|im_end|>
    <|im_start|>user\n{content}<|im_end|>
    <|im_start|>assistant\n{content}<|im_end|>
    
    Used by: GPT-3.5-turbo, GPT-4, many open models
    """
    
    @classmethod
    def format(cls, messages: List[Dict[str, str]]) -> str:
        """Format as ChatML."""
        formatted = ""
        for msg in messages:
            role = msg["role"]
            content = msg["content"]
            formatted += f"<|im_start|>{role}\n{content}<|im_end|>\n"
        return formatted


# Demonstrate formats
example = InstructionExample(
    instruction="Summarize the following article in one sentence.",
    input="Machine learning is a subset of artificial intelligence...",
    output="Machine learning is an AI technique for learning from data."
)

print("=" * 80)
print("ALPACA FORMAT")
print("=" * 80)
print(AlpacaFormat.format(example))

print("\n" + "=" * 80)
print("DOLLY FORMAT")
print("=" * 80)
print(DollyFormat.format(
    instruction=example.instruction,
    context=example.input,
    response=example.output
))

print("\n" + "=" * 80)
print("SHAREGPT FORMAT (Multi-turn)")
print("=" * 80)
messages = [
    {"role": "user", "content": "What is machine learning?"},
    {"role": "assistant", "content": "Machine learning is a subset of AI..."},
    {"role": "user", "content": "Can you give an example?"},
    {"role": "assistant", "content": "Sure! Image classification is..."},
]
print(ShareGPTFormat.format(messages, system_prompt="You are a helpful AI assistant."))

print("\n" + "=" * 80)
print("CHATML FORMAT")
print("=" * 80)
print(ChatMLFormat.format([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi! How can I help you today?"},
]))

## 2. Dataset Preparation

In [ ]:
class InstructionDataset(Dataset):
    """Dataset for instruction tuning.
    
    Handles tokenization and formatting for instruction examples.
    """
    
    def __init__(self, 
                 examples: List[Dict],
                 tokenizer,
                 max_length: int = 2048,
                 format_style: str = 'alpaca'):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.format_style = format_style
    
    def __len__(self) -> int:
        return len(self.examples)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        example = self.examples[idx]
        
        # Format based on style
        if self.format_style == 'alpaca':
            formatted = self._format_alpaca(example)
        elif self.format_style == 'sharegpt':
            formatted = self._format_sharegpt(example)
        else:
            raise ValueError(f"Unknown format: {self.format_style}")
        
        # Tokenize
        # Note: In real implementation, use actual tokenizer
        # Here we simulate with random tokens
        tokens = list(range(np.random.randint(100, 512)))
        
        # Pad/truncate
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]
        
        # Create attention mask (1 for real tokens, 0 for padding)
        attention_mask = [1] * len(tokens)
        
        # Pad to max_length
        padding_length = self.max_length - len(tokens)
        tokens.extend([0] * padding_length)
        attention_mask.extend([0] * padding_length)
        
        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(tokens, dtype=torch.long),  # For causal LM
        }
    
    def _format_alpaca(self, example: Dict) -> str:
        """Format as Alpaca-style."""
        instruction = example.get('instruction', '')
        input_text = example.get('input', '')
        output = example.get('output', '')
        
        ex = InstructionExample(instruction, input_text, output)
        return AlpacaFormat.format(ex)
    
    def _format_sharegpt(self, example: Dict) -> str:
        """Format as ShareGPT-style."""
        messages = example.get('conversations', [])
        return ShareGPTFormat.format(messages)


# Example: Create synthetic instruction dataset
def create_synthetic_dataset(num_examples: int = 100) -> List[Dict]:
    """Generate synthetic instruction examples."""
    
    templates = [
        {
            'instruction': 'Summarize the following text.',
            'input': 'Sample text about topic X...',
            'output': 'Topic X is about...'
        },
        {
            'instruction': 'Translate the following to French.',
            'input': 'Hello, how are you?',
            'output': 'Bonjour, comment allez-vous?'
        },
        {
            'instruction': 'Write a poem about nature.',
            'input': '',
            'output': 'Trees sway in the breeze...'
        },
        {
            'instruction': 'Explain the concept of recursion.',
            'input': '',
            'output': 'Recursion is when a function calls itself...'
        },
    ]
    
    dataset = []
    for i in range(num_examples):
        template = templates[i % len(templates)]
        dataset.append(template.copy())
    
    return dataset


# Demonstrate dataset creation
examples = create_synthetic_dataset(100)
print(f"Created {len(examples)} instruction examples")
print(f"\nExample:")
print(json.dumps(examples[0], indent=2))

## 3. System Prompts and Chat Templates

System prompts guide model behavior

In [ ]:
class SystemPrompts:
    """Common system prompts for different use cases."""
    
    # General assistant
    HELPFUL_ASSISTANT = (
        "You are a helpful, respectful and honest assistant. "
        "Always answer as helpfully as possible, while being safe. "
        "If you don't know the answer to a question, please don't share false information."
    )
    
    # Technical assistant
    TECHNICAL_ASSISTANT = (
        "You are a technical assistant specialized in programming and software engineering. "
        "Provide clear, accurate code examples and explanations. "
        "When showing code, include comments to explain key concepts."
    )
    
    # Creative writing
    CREATIVE_WRITER = (
        "You are a creative writing assistant. "
        "Help users craft engaging stories, poems, and other creative content. "
        "Be imaginative while following the user's guidelines."
    )
    
    # Educational tutor
    TUTOR = (
        "You are an educational tutor. "
        "Explain concepts clearly using examples and analogies. "
        "Break down complex topics into digestible parts. "
        "Encourage learning through questions and practice."
    )
    
    # Research assistant
    RESEARCH_ASSISTANT = (
        "You are a research assistant. "
        "Provide well-researched, accurate information with proper context. "
        "Cite sources when possible and acknowledge uncertainties."
    )


class ChatTemplate:
    """Apply chat templates for different model formats."""
    
    @staticmethod
    def apply_llama2_template(messages: List[Dict[str, str]]) -> str:
        """Apply LLaMA-2 chat template.
        
        Format:
            <s>[INST] <<SYS>>\n{system}\n<</SYS>>\n\n{user} [/INST] {assistant} </s>
        """
        formatted = ""
        system_msg = None
        
        # Extract system message if present
        if messages[0]["role"] == "system":
            system_msg = messages[0]["content"]
            messages = messages[1:]
        
        # Format conversation
        for i in range(0, len(messages), 2):
            if i == 0 and system_msg:
                formatted += f"<s>[INST] <<SYS>>\n{system_msg}\n<</SYS>>\n\n"
            else:
                formatted += "<s>[INST] "
            
            formatted += messages[i]["content"] + " [/INST] "
            
            if i + 1 < len(messages):
                formatted += messages[i + 1]["content"] + " </s>"
        
        return formatted
    
    @staticmethod
    def apply_mistral_template(messages: List[Dict[str, str]]) -> str:
        """Apply Mistral chat template.
        
        Format:
            <s>[INST] {user} [/INST] {assistant}</s>[INST] {user} [/INST]
        """
        formatted = ""
        
        for i, msg in enumerate(messages):
            if msg["role"] == "user":
                if i == 0:
                    formatted += f"<s>[INST] {msg['content']} [/INST]"
                else:
                    formatted += f"[INST] {msg['content']} [/INST]"
            elif msg["role"] == "assistant":
                formatted += f" {msg['content']}</s>"
        
        return formatted
    
    @staticmethod
    def apply_vicuna_template(messages: List[Dict[str, str]]) -> str:
        """Apply Vicuna chat template.
        
        Format:
            USER: {user}\nASSISTANT: {assistant}\nUSER: ...
        """
        formatted = ""
        
        for msg in messages:
            role = msg["role"].upper()
            if role == "SYSTEM":
                formatted += f"{msg['content']}\n\n"
            else:
                formatted += f"{role}: {msg['content']}\n"
        
        return formatted.strip()


# Demonstrate templates
messages = [
    {"role": "system", "content": SystemPrompts.HELPFUL_ASSISTANT},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user", "content": "What's the population?"},
]

print("=" * 80)
print("LLAMA-2 CHAT TEMPLATE")
print("=" * 80)
print(ChatTemplate.apply_llama2_template(messages.copy()))

print("\n" + "=" * 80)
print("MISTRAL CHAT TEMPLATE")
print("=" * 80)
print(ChatTemplate.apply_mistral_template(messages[1:]))  # Mistral doesn't use system in same way

print("\n" + "=" * 80)
print("VICUNA CHAT TEMPLATE")
print("=" * 80)
print(ChatTemplate.apply_vicuna_template(messages.copy()))

## 4. Training Loop for Instruction Tuning

In [ ]:
class InstructionTuningTrainer:
    """Trainer for instruction tuning.
    
    Key differences from pre-training:
    1. Train only on assistant responses (mask user prompts)
    2. Typically shorter training (few epochs)
    3. Lower learning rate
    4. More careful evaluation
    """
    
    def __init__(self, model, tokenizer, config: Dict):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
    
    def train(self, train_dataset: Dataset, eval_dataset: Dataset):
        """Train with instruction tuning."""
        
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.config.get('learning_rate', 2e-5)
        )
        
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.config.get('batch_size', 4),
            shuffle=True,
        )
        
        num_epochs = self.config.get('num_epochs', 3)
        
        train_losses = []
        
        self.model.train()
        for epoch in range(num_epochs):
            epoch_loss = 0
            
            for batch_idx, batch in enumerate(train_loader):
                # Forward pass (simplified)
                input_ids = batch['input_ids']
                attention_mask = batch['attention_mask']
                labels = batch['labels']
                
                # In real training, would use model forward pass
                # Here we simulate loss
                loss = torch.tensor(5.0 - epoch - batch_idx * 0.01)
                
                # Backward
                optimizer.zero_grad()
                # loss.backward()  # Would do in real training
                # optimizer.step()
                
                epoch_loss += loss.item()
                train_losses.append(loss.item())
                
                if batch_idx % 10 == 0:
                    print(f"Epoch {epoch+1}/{num_epochs} | Batch {batch_idx} | Loss: {loss.item():.4f}")
            
            avg_loss = epoch_loss / len(train_loader)
            print(f"Epoch {epoch+1} completed | Avg Loss: {avg_loss:.4f}")
        
        return train_losses


def mask_prompts_for_training(input_ids: torch.Tensor, 
                               response_start_token: int) -> torch.Tensor:
    """Mask prompt tokens so loss is only computed on responses.
    
    This is crucial for instruction tuning - we only want to train
    the model to generate the assistant's response, not repeat the prompt.
    """
    labels = input_ids.clone()
    
    # Find where response starts
    for i in range(len(input_ids)):
        if input_ids[i] == response_start_token:
            # Mask everything before response
            labels[:i] = -100  # -100 is ignored in loss computation
            break
    
    return labels


# Demonstrate training setup
print("Instruction Tuning Training Setup")
print("="*80)

# Create dummy dataset
examples = create_synthetic_dataset(50)
dataset = InstructionDataset(
    examples,
    tokenizer=None,  # Would use real tokenizer
    max_length=512,
    format_style='alpaca'
)

print(f"Dataset size: {len(dataset)} examples")
print(f"\nTraining configuration:")
config = {
    'learning_rate': 2e-5,
    'batch_size': 4,
    'num_epochs': 3,
    'warmup_steps': 100,
}
for k, v in config.items():
    print(f"  {k}: {v}")

## 5. Evaluation of Instruction Following

In [ ]:
class InstructionEvaluator:
    """Evaluate instruction following capability.
    
    Common evaluation approaches:
    1. Human evaluation (gold standard, expensive)
    2. Model-based evaluation (GPT-4 as judge)
    3. Benchmark datasets (MMLU, Big-Bench, etc.)
    4. Specific task metrics (ROUGE for summarization, etc.)
    """
    
    @staticmethod
    def evaluate_helpfulness(response: str) -> Dict[str, float]:
        """Evaluate helpfulness on multiple dimensions.
        
        Criteria:
        - Relevance: Does it answer the question?
        - Completeness: Is the answer complete?
        - Clarity: Is it easy to understand?
        - Accuracy: Is the information correct?
        """
        # In practice, would use GPT-4 API or human eval
        # Here we simulate scores
        return {
            'relevance': np.random.uniform(0.7, 1.0),
            'completeness': np.random.uniform(0.7, 1.0),
            'clarity': np.random.uniform(0.7, 1.0),
            'accuracy': np.random.uniform(0.7, 1.0),
        }
    
    @staticmethod
    def evaluate_safety(response: str) -> Dict[str, bool]:
        """Evaluate safety aspects.
        
        Check for:
        - Harmful content
        - Biased language
        - Toxic language
        - PII leakage
        """
        # Would use content moderation API in practice
        return {
            'is_safe': True,
            'is_unbiased': True,
            'is_non_toxic': True,
            'no_pii': True,
        }
    
    @staticmethod
    def compare_responses(response_a: str, response_b: str, 
                         instruction: str) -> str:
        """Compare two responses (A/B testing).
        
        Returns: 'A', 'B', or 'tie'
        
        This is how models like GPT-4 were evaluated during RLHF.
        """
        # Would use GPT-4 as judge in practice
        return np.random.choice(['A', 'B', 'tie'], p=[0.4, 0.4, 0.2])


class BenchmarkEvaluator:
    """Evaluate on standard benchmarks."""
    
    BENCHMARKS = {
        'MMLU': 'Massive Multitask Language Understanding (57 tasks)',
        'BBH': 'Big-Bench Hard (23 challenging tasks)',
        'HumanEval': 'Code generation (164 problems)',
        'TruthfulQA': 'Truthfulness evaluation',
        'GSM8K': 'Grade school math (8K problems)',
        'HellaSwag': 'Commonsense reasoning',
        'ARC': 'AI2 Reasoning Challenge',
    }
    
    @classmethod
    def run_benchmark(cls, model, benchmark_name: str) -> float:
        """Run model on benchmark and return score."""
        # Simulate benchmark scores
        base_score = np.random.uniform(0.6, 0.9)
        print(f"\nEvaluating on {benchmark_name}...")
        print(f"  Description: {cls.BENCHMARKS[benchmark_name]}")
        print(f"  Score: {base_score:.2%}")
        return base_score


# Demonstrate evaluation
print("Instruction Following Evaluation")
print("="*80)

# Example response
response = "Paris is the capital of France. It is located in northern France..."

# Evaluate helpfulness
evaluator = InstructionEvaluator()
scores = evaluator.evaluate_helpfulness(response)
print("\nHelpfulness Scores:")
for metric, score in scores.items():
    print(f"  {metric.capitalize()}: {score:.2f}")

# Safety check
safety = evaluator.evaluate_safety(response)
print("\nSafety Checks:")
for check, passed in safety.items():
    status = "PASS" if passed else "FAIL"
    print(f"  {check}: {status}")

# Benchmark evaluation
print("\n" + "="*80)
print("Benchmark Evaluation")
print("="*80)

benchmark_eval = BenchmarkEvaluator()
benchmark_scores = {}
for benchmark in ['MMLU', 'BBH', 'HumanEval', 'GSM8K']:
    score = benchmark_eval.run_benchmark(None, benchmark)
    benchmark_scores[benchmark] = score

# Plot benchmark scores
plt.figure(figsize=(10, 6))
benchmarks = list(benchmark_scores.keys())
scores = [benchmark_scores[b] for b in benchmarks]
bars = plt.bar(benchmarks, scores, edgecolor='black', alpha=0.7)
plt.ylabel('Score', fontsize=12)
plt.title('Instruction-Tuned Model Benchmark Performance', fontsize=14, fontweight='bold')
plt.ylim([0, 1])
plt.grid(True, alpha=0.3, axis='y')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1%}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('instruction_tuning_benchmarks.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Best Practices for Instruction Tuning

In [ ]:
class InstructionTuningBestPractices:
    """Best practices learned from Alpaca, Vicuna, and other projects."""
    
    @staticmethod
    def data_quality_tips():
        """Tips for high-quality instruction data."""
        tips = [
            "Diversity: Include wide range of tasks and domains",
            "Quality > Quantity: 10K high-quality > 100K low-quality",
            "Balance: Mix of open-ended and specific tasks",
            "Multi-turn: Include conversations, not just single QA",
            "Negative examples: Include what NOT to do",
            "Difficulty range: Easy to hard examples",
            "Real-world: Use actual user queries when possible",
            "Safety: Filter toxic/harmful content",
        ]
        return tips
    
    @staticmethod
    def training_tips():
        """Training hyperparameter recommendations."""
        return {
            'learning_rate': '2e-5 to 1e-4 (lower than pre-training)',
            'batch_size': '4-8 per GPU (with gradient accumulation)',
            'epochs': '3-5 (more risks overfitting)',
            'warmup_ratio': '0.03-0.1',
            'weight_decay': '0.0-0.1',
            'gradient_clipping': '1.0',
            'sequence_length': '2048-4096',
            'lr_schedule': 'cosine or linear decay',
        }
    
    @staticmethod
    def data_mixing_strategy():
        """How to mix different data sources."""
        return {
            'General QA': '40%',
            'Chat/Dialogue': '25%',
            'Code': '15%',
            'Math/Reasoning': '10%',
            'Creative Writing': '5%',
            'Safety/Alignment': '5%',
        }
    
    @staticmethod
    def common_pitfalls():
        """Common mistakes to avoid."""
        return [
            "Training too long (overfitting on instruction data)",
            "Learning rate too high (catastrophic forgetting)",
            "Ignoring format consistency (mix different templates)",
            "Not masking prompts (model learns to repeat instructions)",
            "Insufficient diversity (model only learns narrow behavior)",
            "No safety filtering (model learns toxic patterns)",
            "Forgetting to evaluate (no idea if it actually works better)",
        ]


# Display best practices
bp = InstructionTuningBestPractices()

print("INSTRUCTION TUNING BEST PRACTICES")
print("="*80)

print("\n1. DATA QUALITY TIPS:")
for tip in bp.data_quality_tips():
    print(f"  - {tip}")

print("\n2. TRAINING HYPERPARAMETERS:")
for param, value in bp.training_tips().items():
    print(f"  {param}: {value}")

print("\n3. DATA MIXING STRATEGY:")
mixing = bp.data_mixing_strategy()
for source, ratio in mixing.items():
    print(f"  {source}: {ratio}")

# Visualize data mixing
plt.figure(figsize=(10, 6))
sources = list(mixing.keys())
ratios = [float(r.strip('%')) for r in mixing.values()]
colors = plt.cm.Set3(range(len(sources)))
plt.pie(ratios, labels=sources, autopct='%1.0f%%', colors=colors, startangle=90)
plt.title('Recommended Data Mixing Strategy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data_mixing_strategy.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n4. COMMON PITFALLS TO AVOID:")
for pitfall in bp.common_pitfalls():
    print(f"  - {pitfall}")

## Key Takeaways

### What is Instruction Tuning?
- Fine-tune pre-trained model on (instruction, response) pairs
- Teaches model to follow instructions rather than just complete text
- Typically requires 10K-100K high-quality examples
- Much shorter training than pre-training (3-5 epochs)

### Common Formats
1. **Alpaca**: Stanford format with clear ### markers
2. **ShareGPT**: Multi-turn conversations from ChatGPT shares
3. **Dolly**: Databricks format with categories
4. **ChatML**: OpenAI-style with special tokens

### Data Requirements
- **Quality > Quantity**: 10K good examples > 100K bad
- **Diversity**: Wide range of tasks and domains
- **Multi-turn**: Not just single Q&A
- **Safety**: Filter toxic/harmful content

### Training Tips
- Lower LR than pre-training (2e-5 to 1e-4)
- 3-5 epochs (more risks overfitting)
- Mask prompts (only compute loss on responses)
- Monitor for catastrophic forgetting

### Evaluation
- Human eval (gold standard)
- Model-as-judge (GPT-4 evaluation)
- Benchmarks (MMLU, BBH, etc.)
- A/B testing

### Historical Impact
- **2021**: FLAN showed instruction tuning improves zero-shot
- **2023**: Alpaca showed 52K examples sufficient for 7B model
- **2023**: Vicuna showed ShareGPT data works well
- **2024-2025**: Standard practice for all LLM releases

### Modern Practice
- Most open models now include instruction-tuned variants
- Often followed by RLHF/DPO for further alignment
- Critical step between base model and production assistant